# CA1 Two-Compartment LIF: Axial Coupling, Dendritic Ih & Resonance

*October 26th, 2026*

Investigating dendrite-to-soma EPSP propagation and frequency-dependent responses in a reduced CA1 pyramidal neuron model, informed by Lowet et al. (2023).



## Scope and parameter provenance
This is your existing exploratory two-compartment LIF + dendritic Ih model,
cleaned for sequential execution. It retains your geometry, currents, kinetics,
and colors. These effective parameters are not a fitted reconstruction of Lowet
or an independently verified Ceballos parameter set. HCN kinetics here differ
from the original HHca2 notebook.

Units: ms, mV, nF, µS, nA. Only the soma resets; no refractory interval is imposed.
Holding current matches the resting voltage between Ih conditions. Frequency
curves measure dendritic-current-to-somatic-voltage transfer impedance.
EPSP peak ratios and sinusoidal amplitude ratios are different measurements.
Run all cells from a fresh kernel. Dependencies: numpy, pandas, matplotlib, IPython.
Results save to a local folder at the end; no Google Drive mount is required.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Time
dt = 0.1                # ms
t_max = 150.0           # ms
time = np.arange(0, t_max + dt/2, dt)

# Compartment geometry
p = 0.15                # Somatic area fraction
A_total = 1e-4          # cm², illustrative
A_s = p * A_total
A_d = (1 - p) * A_total

# Membrane densities
Cm_density = 1.0        # µF/cm²
gL_density = 0.18       # mS/cm²

# Absolute capacitances (nF)
C_s = Cm_density * A_s * 1000
C_d = Cm_density * A_d * 1000

# Absolute leak conductances (µS)
g_Ls = gL_density * A_s * 1000
g_Ld = gL_density * A_d * 1000

# Membrane potentials (mV)
E_L = -65.0
V_th = -50.0
V_reset = -65.0

# Axial conductance (µS)
g_ax = 0.01

# AMPA-like dendritic synaptic input
E_syn = 0.0
tau_syn = 2.0           # ms
g_syn_max = 0.015       # µS
t_stim = 30.0           # ms

# Dendritic Ih
E_h = -30.0             # mV
g_h_active = 0.05       # µS

# Plot colors
PINK = "#FF1493"
LIGHT_PINK = "#F4A6C1"
ORANGE = "#FF8C42"
LIGHT_ORANGE = "#FFBE80"
PURPLE = "#9B59B6"
GREEN = "#59A88F"

print(f"Soma capacitance: {C_s:.4f} nF")
print(f"Dendrite capacitance: {C_d:.4f} nF")
print(f"Soma leak: {g_Ls:.5f} µS")
print(f"Dendrite leak: {g_Ld:.5f} µS")
print(f"Axial coupling: {g_ax:.4f} µS")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
def h_inf(v):
    """Steady-state HCN activation."""
    return 1.0 / (1.0 + np.exp((v + 75.0) / 5.5))


def tau_h(v):
    """HCN activation time constant (ms)."""
    return 20.0 + 100.0 / (
        1.0 + np.exp((v + 75.0) / 10.0)
    )


def simulate_ca1(
    g_h=0.0,
    g_ax_value=None,
    stimulus="epsp",
    frequency=8.0,
    amplitude=0.001,
    duration=None,
    hold_rest=False,
    theta_hz=9.0,
    gamma_hz=40.0,
    tau_scale=1.0
):
    """
    Two-compartment CA1 LIF with dendritic Ih.

    Stimuli:
        'epsp'        - dendritic AMPA-like synaptic input
        'sine'        - single-frequency sinusoidal current
        'theta'       - theta-frequency current
        'gamma'       - gamma-frequency current
        'theta_gamma' - combined theta and gamma currents
        'none'        - no external stimulation

    All current inputs are applied to the dendrite.
    """

    if tau_scale <= 0 or g_h < 0:
        raise ValueError('tau_scale must be positive and g_h nonnegative')
    if stimulus not in ('epsp', 'sine', 'theta', 'gamma', 'theta_gamma', 'none'):
        raise ValueError('Unknown stimulus')
    if g_ax_value is None:
        g_ax_value = g_ax

    if duration is None:
        duration = (
            t_max if stimulus in ("epsp", "none")
            else 3000.0
        )

    if g_ax_value < 0 or duration <= 0:
        raise ValueError('Coupling must be nonnegative and duration positive')
    t = np.arange(0, duration + dt/2, dt)
    n = len(t)

    # Recorded variables
    Vs = np.zeros(n)
    Vd = np.zeros(n)
    h = np.zeros(n)
    g_syn = np.zeros(n)
    I_ax_trace = np.zeros(n)
    I_ext_trace = np.zeros(n)

    spikes = []

    # Initial conditions
    vs = E_L
    vd = E_L
    gate = h_inf(E_L)

    # Holding current offsets resting Ih
    I_hold = (
        g_h * h_inf(E_L) * (E_L - E_h)
        if hold_rest else 0.0
    )

    # Membrane equations
    def derivatives(vs, vd, gate, I_syn, I_ext):

        I_h = g_h * gate * (vd - E_h)
        I_ax = g_ax_value * (vd - vs)

        # Soma
        d_vs = (
            -g_Ls * (vs - E_L)
            + I_ax
        ) / C_s

        # Dendrite
        d_vd = (
            -g_Ld * (vd - E_L)
            - I_syn
            - I_h
            - I_ax
            + I_ext
            + I_hold
        ) / C_d

        # HCN gating
        d_h = (
            h_inf(vd) - gate
        ) / (tau_scale * tau_h(vd))

        return d_vs, d_vd, d_h, I_ax

    # Equilibrate membrane for 1000 ms
    for _ in range(int(1000 / dt)):

        dvs, dvd, dh, _ = derivatives(
            vs, vd, gate, 0.0, 0.0
        )

        vs += dt * dvs
        vd += dt * dvd
        gate += dt * dh

    Vs[0] = vs
    Vd[0] = vd
    h[0] = gate

    # Main simulation
    for i in range(1, n):

        vs = Vs[i-1]
        vd = Vd[i-1]
        gate = h[i-1]

        I_syn = 0.0
        I_ext = 0.0

        # AMPA-like EPSP
        if stimulus == "epsp" and t[i] >= t_stim:

            g_syn[i] = g_syn_max * np.exp(
                -(t[i] - t_stim) / tau_syn
            )

            I_syn = g_syn[i] * (vd - E_syn)

        # Single-frequency sine wave
        elif stimulus == "sine":

            I_ext = amplitude * np.sin(
                2 * np.pi * frequency * t[i] / 1000
            )

        # Theta
        elif stimulus == "theta":

            I_ext = amplitude * np.sin(
                2 * np.pi * theta_hz * t[i] / 1000
            )

        # Gamma
        elif stimulus == "gamma":

            I_ext = amplitude * np.sin(
                2 * np.pi * gamma_hz * t[i] / 1000
            )

        # Combined theta + gamma
        elif stimulus == "theta_gamma":

            theta = np.sin(
                2 * np.pi * theta_hz * t[i] / 1000
            )

            gamma = np.sin(
                2 * np.pi * gamma_hz * t[i] / 1000
            )

            I_ext = amplitude * (theta + gamma)

        # Update membrane states
        dvs, dvd, dh, I_ax = derivatives(
            vs, vd, gate, I_syn, I_ext
        )

        Vs[i] = vs + dt * dvs
        Vd[i] = vd + dt * dvd
        h[i] = gate + dt * dh

        I_ax_trace[i] = I_ax
        I_ext_trace[i] = I_ext

        # Somatic spike threshold/reset
        if Vs[i] >= V_th:

            spikes.append(t[i])
            Vs[i] = V_reset

    return {
        "time": t,
        "Vs": Vs,
        "Vd": Vd,
        "h": h,
        "g_syn": g_syn,
        "I_ax": I_ax_trace,
        "I_ext": I_ext_trace,
        "tau_scale": tau_scale,
        "spikes": spikes
    }


def delta_v(result, compartment):
    return (
        result[compartment]
        - result[compartment][0]
    )

In [ ]:
# Same resting voltage for both conditions
passive = simulate_ca1(g_h=0.0, hold_rest=True)
active = simulate_ca1(g_h=g_h_active, hold_rest=True)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    time, delta_v(passive, "Vd"),
    color=LIGHT_PINK, ls="--", lw=2,
    label="Dendrite — Ih OFF"
)

ax.plot(
    time, delta_v(active, "Vd"),
    color=PINK, lw=2.5,
    label="Dendrite — Ih ON"
)

ax.plot(
    time, delta_v(passive, "Vs"),
    color=LIGHT_ORANGE, ls="--", lw=2,
    label="Soma — Ih OFF"
)

ax.plot(
    time, delta_v(active, "Vs"),
    color=ORANGE, lw=2.5,
    label="Soma — Ih ON"
)

ax.axvline(t_stim, color="gray", ls=":")
ax.set(
    title="CA1 — Dendrite-to-Soma EPSP Propagation",
    xlabel="Time (ms)",
    ylabel="Change in voltage (mV)"
)
ax.set_xlim(20, 120)
ax.legend(frameon=False)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()

In [ ]:
def epsp_metrics(result):
    if result['spikes']:
        return {key: np.nan for key in [
            'Dendritic EPSP (mV)', 'Somatic EPSP (mV)', 'Transfer ratio',
            'Attenuation (%)', 'Dendritic peak (ms)', 'Somatic peak (ms)',
            'Propagation delay (ms)']}
    t = result["time"]
    mask = t >= t_stim

    dend = delta_v(result, "Vd")
    soma = delta_v(result, "Vs")

    dend_peak = np.max(dend[mask])
    soma_peak = np.max(soma[mask])

    td = t[mask][np.argmax(dend[mask])]
    ts = t[mask][np.argmax(soma[mask])]

    return {
        "Dendritic EPSP (mV)": dend_peak,
        "Somatic EPSP (mV)": soma_peak,
        "Transfer ratio": soma_peak / dend_peak,
        "Attenuation (%)": 100 * (
            1 - soma_peak / dend_peak
        ),
        "Dendritic peak (ms)": td,
        "Somatic peak (ms)": ts,
        "Propagation delay (ms)": ts - td
    }


comparison = pd.DataFrame({
    "Ih OFF": epsp_metrics(passive),
    "Ih ON": epsp_metrics(active)
}).T

display(comparison.round(3))

## Experiment: Axial Coupling × Dendritic Ih
**Question:** How does \(g_{ax}\) influence the effect of Ih on dendrite-to-soma EPSP propagation?

In [ ]:
g_ax_values = [0.001, 0.003, 0.005, 0.01, 0.02, 0.04]
g_h_values = [0.00, 0.01, 0.03, 0.05, 0.08]

results = []

for ax_value in g_ax_values:
    for gh in g_h_values:

        sim = simulate_ca1(
            g_h=gh,
            g_ax_value=ax_value,
            hold_rest=True
        )

        metrics = epsp_metrics(sim)

        results.append({
            "g_ax": ax_value,
            "g_h": gh,
            **metrics
        })

df = pd.DataFrame(results)

heatmap = df.pivot(
    index="g_h",
    columns="g_ax",
    values="Transfer ratio"
)

plt.figure(figsize=(9, 5))
plt.imshow(
    heatmap,
    origin="lower",
    aspect="auto",
    cmap="RdPu",
    vmin=0,
    vmax=1
)

plt.colorbar(label="Somatic / Dendritic EPSP")
plt.xticks(range(len(g_ax_values)), g_ax_values)
plt.yticks(range(len(g_h_values)), g_h_values)
plt.xlabel("Axial conductance (µS)")
plt.ylabel("Ih conductance (µS)")
plt.title("CA1 — Axial Coupling × Dendritic Ih")
plt.tight_layout()
plt.show()

display(heatmap.round(3))

### Propagation delay heatmap

In [ ]:
delay_map = df.pivot(
    index="g_h",
    columns="g_ax",
    values="Propagation delay (ms)"
)

plt.figure(figsize=(9, 5))

plt.imshow(
    delay_map,
    origin="lower",
    aspect="auto",
    cmap="PuRd"
)

plt.colorbar(label="Somatic peak delay (ms)")
plt.xticks(range(len(g_ax_values)), g_ax_values)
plt.yticks(range(len(g_h_values)), g_h_values)

plt.xlabel("Axial conductance (µS)")
plt.ylabel("Ih conductance (µS)")
plt.title("CA1 — EPSP Propagation Delay")

plt.tight_layout()
plt.show()

display(delay_map.round(3))

Increasing coupling and changing Ih are tested at a matched resting voltage. Interpret transfer and peak delay only for subthreshold trials; these peak delays do not measure axonal conduction.


In [ ]:
# Frequency response settings
frequencies = np.arange(1, 51, 1)  # Hz
ih_values = [0.00, 0.05, 0.08]

# Subthreshold sinusoidal current
sine_amplitude = 0.001  # nA
sine_duration = 4000.0  # ms


def frequency_response(freq, gh, ax_value=g_ax, tau_scale=1.0):

    sim = simulate_ca1(
        g_h=gh,
        g_ax_value=ax_value,
        stimulus="sine",
        frequency=freq,
        amplitude=sine_amplitude,
        duration=sine_duration,
        tau_scale=tau_scale,
        hold_rest=True
    )

    # Resonance analysis requires subthreshold voltage
    if len(sim["spikes"]) > 0:
        return np.nan

    t = sim["time"]
    soma = sim["Vs"]

    # Analyze the final 2000 ms
    mask = t >= 2000.0

    phase = 2 * np.pi * freq * t[mask] / 1000

    X = np.column_stack([
        np.sin(phase),
        np.cos(phase),
        np.ones(mask.sum())
    ])

    coefficients = np.linalg.lstsq(
        X, soma[mask], rcond=None
    )[0]

    voltage_amplitude = np.hypot(
        coefficients[0],
        coefficients[1]
    )

    # mV / nA = MΩ
    return voltage_amplitude / sine_amplitude

In [ ]:
colors = [LIGHT_PINK, ORANGE, PURPLE]

resonance_results = []

plt.figure(figsize=(10, 5))

for gh, color in zip(ih_values, colors):

    response = [
        frequency_response(f, gh)
        for f in frequencies
    ]

    plt.plot(
        frequencies,
        response,
        marker="o",
        color=color,
        linewidth=2.5,
        markersize=5,
        label=f"Ih = {gh} µS"
    )

    for f, z in zip(frequencies, response):
        resonance_results.append({
            "g_h": gh,
            "Frequency (Hz)": f,
            "Transfer impedance (MΩ)": z
        })

plt.xlabel("Input frequency (Hz)")
plt.ylabel("Somatic transfer impedance (MΩ)")
plt.title("CA1 — Subthreshold Frequency Response")
plt.legend(frameon=False)
plt.grid(alpha=0.15)

plt.tight_layout()
plt.show()

resonance_df = pd.DataFrame(resonance_results)

In [ ]:
resonance_summary = []

for gh in ih_values:

    subset = resonance_df[
        resonance_df["g_h"] == gh
    ].dropna()

    if subset.empty:
        continue

    peak_row = subset.loc[
        subset["Transfer impedance (MΩ)"].idxmax()
    ]

    peak_f = peak_row["Frequency (Hz)"]
    peak_z = peak_row["Transfer impedance (MΩ)"]

    # Interior maximum: candidate resonance
    interior_peak = (
        peak_f > frequencies.min()
        and peak_f < frequencies.max()
    )

    resonance_summary.append({
        "Ih (µS)": gh,
        "Peak frequency (Hz)": peak_f,
        "Peak impedance (MΩ)": peak_z,
        "Interior peak": interior_peak
    })

resonance_summary = pd.DataFrame(resonance_summary)

display(resonance_summary.round(3))

In [ ]:
print("MODEL PARAMETERS")
print(f"Soma fraction: {p:.2f}")
print(f"Soma area: {A_s:.2e} cm²")
print(f"Dendrite area: {A_d:.2e} cm²")
print(f"C_s: {C_s:.4f} nF")
print(f"C_d: {C_d:.4f} nF")
print(f"g_Ls: {g_Ls:.5f} µS")
print(f"g_Ld: {g_Ld:.5f} µS")
print(f"g_ax: {g_ax:.4f} µS")

print("\nMEMBRANE TIME CONSTANTS")
print(f"Soma: {C_s/g_Ls:.2f} ms")
print(f"Dendrite: {C_d/g_Ld:.2f} ms")

print("\nEPSP SPIKE CHECK")
print(f"Ih OFF: {len(passive['spikes'])} spikes")
print(f"Ih ON: {len(active['spikes'])} spikes")

print("\nRESONANCE SUMMARY")
display(resonance_summary)

In [ ]:
spike_checks = []

for ax_value in g_ax_values:
    for gh in g_h_values:
        sim = simulate_ca1(
            g_h=gh,
            g_ax_value=ax_value,
            hold_rest=True
        )
        spike_checks.append({
            "g_ax": ax_value,
            "g_h": gh,
            "Spike count": len(sim["spikes"])
        })

spike_df = pd.DataFrame(spike_checks)
display(spike_df)

if (spike_df["Spike count"] > 0).any():
    print("WARNING: Some EPSP trials crossed spike threshold.")
    print("Exclude these trials from subthreshold attenuation analysis.")
else:
    print("All EPSP sweep trials remained subthreshold.")

In [ ]:
display(resonance_summary.round(3))

Can changing HCN activation kinetics (tau_h) shift the resonance frequency toward the theta band without altering axial coupling?

In [ ]:
# HCN kinetics sweep

tau_scales = [0.5, 1.0, 2.0, 4.0]
test_freqs = np.arange(1, 31, 1)

colors = [PINK, ORANGE, PURPLE, GREEN]
kinetics_results = []

plt.figure(figsize=(10, 5))

try:
    for scale, color in zip(tau_scales, colors):

        response = [
            frequency_response(f, 0.05, ax_value=0.01, tau_scale=scale)
            for f in test_freqs
        ]

        if not np.isfinite(response).any():
            raise ValueError('No subthreshold responses in kinetics sweep')
        peak_idx = np.nanargmax(response)
        peak_freq = test_freqs[peak_idx]

        kinetics_results.append({
            "Tau multiplier": scale,
            "Peak frequency (Hz)": peak_freq,
            "Peak impedance (MΩ)": response[peak_idx]
        })

        plt.plot(test_freqs, response,
                 color=color, lw=2.5,
                 label=f"τh × {scale}")

finally:
    pass  # No global kinetics state was changed.

plt.axvspan(4, 12, color=LIGHT_PINK,
            alpha=0.15, label="Theta band")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Somatic transfer impedance (MΩ)")
plt.title("CA1 — HCN Kinetics and Resonance")
plt.legend()
plt.grid(alpha=0.15)
plt.tight_layout()
plt.show()

kinetics_df = pd.DataFrame(kinetics_results)
display(kinetics_df.round(3))

The kinetics sweep tests whether slower HCN activation shifts the transfer-impedance peak toward theta. Read the computed table before drawing a conclusion; an edge maximum does not establish resonance. The kinetic multipliers are exploratory choices.


In [ ]:
# The shared simulator above supports all rhythmic stimuli.
# tau_scale is passed explicitly for each experiment.


In [ ]:
# HCN kinetics
tau_scale = 4.0

# Fixed conductances
g_ax = 0.01
g_h = 0.05

conditions = ["theta", "gamma", "theta_gamma"]

rhythmic_results = {}

for condition in conditions:
    rhythmic_results[condition] = simulate_ca1(
        g_h=g_h,
        g_ax_value=g_ax,
        stimulus=condition,
        amplitude=0.001,
        duration=2000,
        tau_scale=4.0,
        hold_rest=True
    )

print("All three simulations completed.")

In [ ]:
#Theta - 9 Hz

sim = rhythmic_results["theta"]
t = sim["time"]

fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

ax[0].plot(t, sim["I_ext"], color=PINK)
ax[0].set_ylabel("Input (nA)")

ax[1].plot(t, delta_v(sim, "Vd"), color=PINK)
ax[1].set_ylabel("Dendrite (mV)")

ax[2].plot(t, delta_v(sim, "Vs"), color=ORANGE)
ax[2].set_ylabel("Soma (mV)")
ax[2].set_xlabel("Time (ms)")

for a in ax:
    a.set_xlim(1000, 1500)
    a.grid(alpha=0.15)

fig.suptitle("CA1 — Theta Input (9 Hz)")
plt.tight_layout()
plt.show()

In [ ]:
# gamma - 40 Hz

sim = rhythmic_results["gamma"]
t = sim["time"]

fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

ax[0].plot(t, sim["I_ext"], color=PURPLE)
ax[0].set_ylabel("Input (nA)")

ax[1].plot(t, delta_v(sim, "Vd"), color=PINK)
ax[1].set_ylabel("Dendrite (mV)")

ax[2].plot(t, delta_v(sim, "Vs"), color=ORANGE)
ax[2].set_ylabel("Soma (mV)")
ax[2].set_xlabel("Time (ms)")

for a in ax:
    a.set_xlim(1000, 1500)
    a.grid(alpha=0.15)

fig.suptitle("CA1 — Gamma Input (40 Hz)")
plt.tight_layout()
plt.show()

In [ ]:
# theta + gamma

sim = rhythmic_results["theta_gamma"]
t = sim["time"]

fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

ax[0].plot(t, sim["I_ext"], color=PURPLE)
ax[0].set_ylabel("Input (nA)")

ax[1].plot(t, delta_v(sim, "Vd"), color=PINK)
ax[1].set_ylabel("Dendrite (mV)")

ax[2].plot(t, delta_v(sim, "Vs"), color=ORANGE)
ax[2].set_ylabel("Soma (mV)")
ax[2].set_xlabel("Time (ms)")

for a in ax:
    a.set_xlim(1000, 1500)
    a.grid(alpha=0.15)

fig.suptitle("CA1 — Combined Theta + Gamma")
plt.tight_layout()
plt.show()

# analyze 9Hz only
**Question**: Does the soma show a smaller, delayed voltage response compared with the dendrite?

In [ ]:
sim = rhythmic_results["theta"]
t = sim["time"]

# Analyze steady-state response
mask = t >= 1000

vd = sim["Vd"][mask]
vs = sim["Vs"][mask]

# Voltage oscillation amplitudes
amp_d = (vd.max() - vd.min()) / 2
amp_s = (vs.max() - vs.min()) / 2

# Dendrite-to-soma transfer
transfer = amp_s / amp_d
attenuation = 100 * (1 - transfer)

print(f"Dendritic amplitude: {amp_d:.4f} mV")
print(f"Somatic amplitude:   {amp_s:.4f} mV")
print(f"Transfer ratio:      {transfer:.3f}")
print(f"Attenuation:         {attenuation:.1f}%")

**Question**: Does gamma (40 Hz) experience greater attenuation than theta (9 Hz)?

In [ ]:
sim = rhythmic_results["gamma"]
t = sim["time"]

# Analyze steady-state response
mask = t >= 1000

vd = sim["Vd"][mask]
vs = sim["Vs"][mask]

# Voltage oscillation amplitudes
amp_d = (vd.max() - vd.min()) / 2
amp_s = (vs.max() - vs.min()) / 2

# Dendrite-to-soma transfer
transfer = amp_s / amp_d
attenuation = 100 * (1 - transfer)

print(f"Dendritic amplitude: {amp_d:.4f} mV")
print(f"Somatic amplitude:   {amp_s:.4f} mV")
print(f"Transfer ratio:      {transfer:.3f}")
print(f"Attenuation:         {attenuation:.1f}%")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rhythmic_rows = []
for condition, frequency in [('theta', 9.0), ('gamma', 40.0)]:
    result = rhythmic_results[condition]
    if result['spikes']:
        raise ValueError('Rhythmic amplitude analysis requires subthreshold trials')
    mask = result['time'] >= 1000.0
    phase = 2*np.pi*frequency*result['time'][mask]/1000
    X = np.column_stack([np.sin(phase), np.cos(phase), np.ones(mask.sum())])
    amplitudes = []
    phases = []
    for compartment in ['Vd', 'Vs']:
        coeff = np.linalg.lstsq(X, result[compartment][mask], rcond=None)[0]
        amplitudes.append(np.hypot(coeff[0], coeff[1]))
        phases.append(np.arctan2(coeff[1], coeff[0]))
    dend_amp, soma_amp = amplitudes
    phase_lag = (phases[0]-phases[1]+np.pi) % (2*np.pi)-np.pi
    rhythmic_rows.append({
        'Frequency': f'{condition.title()} ({frequency:g} Hz)',
        'Dendrite': dend_amp, 'Soma': soma_amp,
        'Attenuation': 100*(1-soma_amp/dend_amp),
        'Soma phase lag (degrees)': np.degrees(phase_lag),
        'Phase delay (ms)': phase_lag/(2*np.pi*frequency)*1000
    })
rhythmic_comparison = pd.DataFrame(rhythmic_rows)

x = np.arange(2)
width = 0.32

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Voltage amplitudes
ax[0].bar(x - width/2, rhythmic_comparison["Dendrite"],
          width, color="#FF1493", label="Dendrite")
ax[0].bar(x + width/2, rhythmic_comparison["Soma"],
          width, color="darkgreen", label="Soma")

ax[0].set_xticks(x, rhythmic_comparison["Frequency"])
ax[0].set_ylabel("Voltage amplitude (mV)")
ax[0].set_title("Theta vs. Gamma — Voltage Response")
ax[0].legend(frameon=False)

# Attenuation
ax[1].bar(rhythmic_comparison["Frequency"],
          rhythmic_comparison["Attenuation"],
          color=["deeppink", "darkgreen"])

ax[1].set_ylabel("Attenuation (%)")
ax[1].set_title("Dendrite-to-Soma Attenuation")

for a in ax:
    a.grid(axis="y", alpha=0.15)
    a.set_axisbelow(True)

plt.tight_layout()
plt.show()

display(rhythmic_comparison)

In [ ]:
from pathlib import Path
output_dir = Path('ca1_resonance_results')
output_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(output_dir / 'ih_epsp_comparison.csv')
df.to_csv(output_dir / 'axial_ih_sweep.csv', index=False)
resonance_df.to_csv(output_dir / 'frequency_response.csv', index=False)
resonance_summary.to_csv(output_dir / 'resonance_summary.csv', index=False)
spike_df.to_csv(output_dir / 'spike_checks.csv', index=False)
kinetics_df.to_csv(output_dir / 'kinetics_summary.csv', index=False)
rhythmic_comparison.to_csv(output_dir / 'theta_gamma_comparison.csv', index=False)
for condition, result in rhythmic_results.items():
    pd.DataFrame({key: result[key] for key in
                  ['time', 'Vs', 'Vd', 'h', 'I_ax', 'I_ext']}).to_csv(
                      output_dir / f'{condition}_trace.csv', index=False)
print('Saved results to:', output_dir.resolve())
